# 32 — Evaluating explanations: the deletion test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/032-evaluating-explanations.ipynb)

An explanation is a promise: the features it ranks highest are the ones the model leans on. The deletion test keeps that promise honest by removing the top-ranked features and measuring what happens to accuracy. A real explanation loses accuracy fast; a random one barely moves the line.

## 1. The numbers

We build a small supervised problem with 24 features, of which only three carry signal. Knowing which three matters is the whole game; the other 21 are pure noise, and any honest attribution method should rank them near zero.

In [ ]:
import numpy as np                                      # the array library, preinstalled on Colab

rng = np.random.default_rng(0)                           # fixed seed: the data is reproducible
n, d = 600, 24                                           # 600 rows, 24 features

X = rng.normal(size=(n, d))                              # every feature is standard normal noise
true_w = np.zeros(d)                                     # start with all weights at zero
true_w[:3] = [2.0, 1.5, 1.2]                             # only the first three features matter
logit = X @ true_w + 0.3 * rng.normal(size=n)            # the true score, plus label noise
y = (logit > 0).astype(float)                            # binary label: 1 if the score is positive

Xtr, ytr = X[:400], y[:400]                              # the training half
Xte, yte = X[400:], y[400:]                              # the held-out half

print("train", Xtr.shape, " test", Xte.shape)            # 400 rows to fit, 200 to judge
print("share of positives", round(y.mean(), 3))          # roughly balanced
print("true weights, first three", true_w[:3])           # 2.0, 1.5, 1.2
print("true weights that are zero:", int((true_w == 0).sum()))   # 21 of 24

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, the array library Colab already has.</li><li><code>rng = ...</code> — a seeded generator, so every run sees the same data.</li><li><code>n, d = 600, 24</code> — rows and features.</li><li><code>X = rng.normal(...)</code> — the feature matrix, all noise until we weight it.</li><li><code>true_w[:3] = ...</code> — put signal in the first three features only.</li><li><code>logit = ...</code> — the real score, with a little label noise added.</li><li><code>y = (logit &gt; 0).astype(float)</code> — threshold the score into a 0/1 label.</li><li><code>Xtr, ytr</code> / <code>Xte, yte</code> — split rows into train and test.</li><li><code>print(...)</code> — confirm the shapes, the class balance, and the sparse truth.</li></ul>

</details>

## 2. Step by step: fit the model

Logistic regression is fitted by gradient descent: compute the predicted probability, compare it with the label, and nudge every weight in the direction that reduces the error. After a few hundred steps the learned weights should concentrate on the three real features.

In [ ]:
def sigmoid(z):                                         # the logistic squash
    return 1 / (1 + np.exp(-z))                          # any real number -> 0 to 1

w = np.zeros(d)                                          # start every weight at zero
b = 0.0                                                  # start the bias at zero

for step in range(800):                                  # 800 gradient steps
    p = sigmoid(Xtr @ w + b)                             # predicted probability for each training row
    grad_w = Xtr.T @ (p - ytr) / len(ytr) + 1e-3 * w     # gradient of the loss, plus a small pull to zero
    grad_b = np.mean(p - ytr)                            # gradient for the bias
    w = w - 0.4 * grad_w                                 # step the weights downhill
    b = b - 0.4 * grad_b                                 # step the bias downhill
    if step % 200 == 0:                                  # report four times, not 800
        loss = -np.mean(ytr * np.log(p + 1e-9) + (1 - ytr) * np.log(1 - p + 1e-9))   # log loss
        print(f"step {step}: loss {loss:.3f}")           # should fall

acc = np.mean((sigmoid(Xte @ w + b) > 0.5) == yte)       # accuracy on rows never seen
print("largest learned weights", np.round(np.sort(np.abs(w))[::-1][:4], 2))   # the signal wins
print("test accuracy", round(acc, 3))                    # about 0.97

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>def sigmoid(z)</code> — squeezes a score into a probability.</li><li><code>w = np.zeros(d)</code> and <code>b = 0.0</code> — the parameters being learned.</li><li><code>p = sigmoid(...)</code> — current predictions on the training rows.</li><li><code>grad_w</code> — the direction in which the weights increase the loss.</li><li><code>w = w - 0.4 * grad_w</code> — move the weights the other way.</li><li><code>if step % 200 == 0</code> — occasional reporting instead of every step.</li><li><code>loss = ...</code> — binary log loss, the number gradient descent is reducing.</li><li><code>acc = ...</code> — threshold the test predictions and compare with the held-out labels.</li><li><code>np.sort(np.abs(w))[::-1][:4]</code> — the four largest learned weights.</li></ul>

</details>

## 3. Plot: delete the top features, watch accuracy fall

The attribution is the model's own weight size. Rank the features by it, delete the top `k`, and re-score the test set. If the ranking is meaningful, accuracy should collapse as soon as the three real features are gone.

In [ ]:
import matplotlib.pyplot as plt                          # plotting library, preinstalled on Colab

def accuracy_without(order, k):                          # accuracy after dropping k features
    dropped = Xte.copy()                                 # never modify the test set itself
    dropped[:, order[:k]] = 0.0                          # zero out the k highest-ranked features
    return np.mean((sigmoid(dropped @ w + b) > 0.5) == yte)   # re-score and compare

order = np.argsort(-np.abs(w))                           # features sorted by learned importance
ks = np.arange(0, 13)                                    # try removing 0 through 12 features
accs = [accuracy_without(order, k) for k in ks]          # one accuracy per k

print("deletion order, first five:", order[:5])          # the three signal features lead
for k, a in zip(ks, accs):                               # print the whole curve
    print(f"removed {k}: accuracy {a:.3f}")              # 0.97 down to chance

plt.figure(figsize=(5, 3))                               # a compact figure
plt.plot(ks, accs, marker="o", color="tab:red")          # accuracy against k
plt.xlabel("number of top features removed")             # x-axis label
plt.ylabel("test accuracy")                              # y-axis label
plt.title("remove the top three and accuracy collapses") # the takeaway
plt.show()                                               # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — the plotting library.</li><li><code>accuracy_without(order, k)</code> — the measurement, reused everywhere below.</li><li><code>dropped = Xte.copy()</code> — a copy, so the original test data survives.</li><li><code>dropped[:, order[:k]] = 0.0</code> — delete by setting the chosen columns to zero.</li><li><code>order = np.argsort(-np.abs(w))</code> — rank features by absolute learned weight.</li><li><code>ks = np.arange(0, 13)</code> — the deletion budgets to test.</li><li><code>accs = [...]</code> — one accuracy number per budget.</li><li><code>plt.plot(...)</code> — accuracy against k; the rest only labels the axes.</li></ul>

</details>

## 4. Interactive: choose how many to delete

Drag the slider to pick `k`. The grey curve is the whole deletion test, the red dot is where you are. Watch the cliff arrive at three and then flatten out: after that, only noise is being deleted.

In [ ]:
from ipywidgets import interact                          # sliders that rerun a function

def show(k=0):                                           # k comes from the slider
    a = accuracy_without(order, k)                       # recompute for this budget only

    plt.figure(figsize=(5, 3))                           # fresh figure for each drag
    plt.plot(ks, accs, color="grey", marker="o", label="deletion curve")   # the reference curve
    plt.scatter([k], [a], color="tab:red", s=80, zorder=3, label="current k")   # mark the chosen point
    plt.xlabel("number of top features removed")         # x-axis label
    plt.ylabel("test accuracy")                          # y-axis label
    plt.title(f"k = {k}: accuracy {a:.3f}")              # live readout
    plt.legend(fontsize=7)                               # name the two marks
    plt.show()                                           # draw it

interact(show, k=(0, 12, 1))                             # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def show(k=0)</code> — the function that redraws one frame.</li><li><code>a = accuracy_without(order, k)</code> — the number for the current slider position.</li><li><code>plt.plot(...)</code> — the fixed reference curve.</li><li><code>plt.scatter(...)</code> — a single red dot for the current k.</li><li><code>plt.title(...)</code> — the readout, so the number is never hidden.</li><li><code>interact(show, k=(...))</code> — the slider, its range and step.</li></ul>

</details>

## 5. When it breaks: a random ranking looks flat

Now replace the attribution with a random ordering and repeat the test. The curve barely moves, because random deletion spreads the damage evenly: with 24 features, removing three at random usually misses the three that matter. A flat deletion curve is the signature of an explanation that carries no information.

In [ ]:
curves = []                                              # one random curve per seed

for seed in range(12):                                   # 12 random rankings
    rand_order = np.random.default_rng(seed).permutation(d)   # a different shuffle each time
    curves.append([accuracy_without(rand_order, int(k)) for k in ks])   # deletion test on that shuffle

curves = np.array(curves)                                # shape: 12 seeds by 13 budgets
mean_curve = curves.mean(axis=0)                         # the average random behaviour

print("random ranking, mean accuracy at k = 3:", round(mean_curve[3], 3))   # about 0.93
print("top-feature ranking, accuracy at k = 3:", round(accs[3], 3))         # about 0.49
print("spread across random rankings at k = 3:", round(curves[:, 3].std(), 3))   # wide, and far from the truth

plt.figure(figsize=(6, 3))                               # wide enough for both curves
plt.plot(ks, accs, color="tab:red", marker="o", label="attribution ranking")   # the real explanation
plt.plot(ks, mean_curve, color="tab:blue", marker="s", label="random ranking (mean)")   # the fake one
plt.fill_between(ks, curves.min(axis=0), curves.max(axis=0), color="tab:blue", alpha=0.2)   # min-max band
plt.ylim(0.3, 1.02)                                      # keep the band on screen
plt.xlabel("number of features removed")                 # x-axis label
plt.ylabel("test accuracy")                              # y-axis label
plt.title("random scores look flat: no cliff, no signal")   # the takeaway
plt.legend(fontsize=7)                                   # name both curves
plt.show()                                               # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>curves = []</code> — collect one curve per random ranking.</li><li><code>rand_order = ...permutation(d)</code> — a random order of the same 24 features.</li><li><code>curves.append([...])</code> — run the identical deletion test on that order.</li><li><code>mean_curve = curves.mean(axis=0)</code> — average accuracy at each budget.</li><li><code>print(...)</code> — contrast k = 3 for the real and the random ranking.</li><li><code>plt.plot(...)</code> — the two curves on one axis.</li><li><code>plt.fill_between(...)</code> — the min-max band: random rankings scatter.</li><li><code>plt.title(...)</code> — names the failure: a flat curve means no signal.</li></ul>

</details>

## Your turn

In section 3, delete features by `order[::-1]` (least important first) and predict the curve before you rerun. Then change the label noise in section 1 from `0.3` to `1.5` and explain why the deletion cliff gets shallower.